# RAG Demo: Filings Question Answering

This notebook demonstrates a read-only Retrieval-Augmented Generation (RAG) pipeline for investment research QA using precomputed filings artifacts. It inspects existing data products, performs deterministic retrieval, and synthesizes a cited answer without recomputing any embeddings or indexes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/filings-qa-rag-demo/blob/main/notebooks/filings_rag_qa.ipynb)


## Colab / Local Setup

This notebook runs both on Google Colab and locally. On Colab it clones the public repository (bringing the `genai_filings` package and the precomputed filings artifacts with it) and installs the Python dependencies. Locally, it detects the existing repo and uses it in place — nothing is cloned or installed.

In [ ]:
# --- Environment detection: Colab vs local ---
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/FranQuant/filings-qa-rag-demo"
REPO_NAME = "filings-qa-rag-demo"

if IN_COLAB:
    # Clone the repo (package + precomputed artifacts) and install deps.
    if not Path(REPO_NAME).exists():
        os.system(f"git clone --depth 1 {REPO_URL}")
    os.chdir(REPO_NAME)
    os.system("pip -q install openai anthropic pyarrow pandas python-dotenv")
    PROJECT_ROOT = Path.cwd()
else:
    # Local: notebook lives in notebooks/, repo root is one level up.
    PROJECT_ROOT = Path("..").resolve()
    os.chdir(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Src path:", SRC_PATH)
print("Working directory:", os.getcwd())

## API Keys

Retrieval embeds your query with OpenAI (the stored vectors use `text-embedding-3-small`), so an **OpenAI key is required**. Answer synthesis can run on OpenAI *or* Anthropic; the **Anthropic key is optional** and only needed for the provider comparison below.

Locally, keys are read from a `.env` file if present. On Colab (or if a key is missing), you'll be prompted to paste it — the value is not stored or echoed.

In [ ]:
import os
from getpass import getpass

# Locally, load .env if it exists (no-op on Colab where there is none).
try:
    from dotenv import load_dotenv
    env_path = PROJECT_ROOT / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print("Loaded environment variables from .env")
except ImportError:
    pass

# OpenAI key is required (for query embeddings + optional generation).
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

# Anthropic key is optional (only for the provider comparison).
if not os.getenv("ANTHROPIC_API_KEY"):
    _ant = getpass(
        "Enter your ANTHROPIC_API_KEY (optional, press Enter to skip): "
    )
    if _ant:
        os.environ["ANTHROPIC_API_KEY"] = _ant

print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))
print("Anthropic key set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## Imports and Artifact Paths

In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq

# Reuse PROJECT_ROOT from environment setup cell (DO NOT redefine)
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "filings" / "NU" / "2025Q2"

sections_path   = DATA_DIR / "sections.parquet"
chunks_path     = DATA_DIR / "chunks.parquet"
embeddings_path = DATA_DIR / "embeddings.parquet"
manifest_path   = DATA_DIR / "embeddings_manifest.json"

## Artifact Sanity Check 

In [ ]:
# PROJECT_ROOT is already defined and cwd is already set correctly
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "filings" / "NU" / "2025Q2"

sections_path   = DATA_DIR / "sections.parquet"
chunks_path     = DATA_DIR / "chunks.parquet"
embeddings_path = DATA_DIR / "embeddings.parquet"
manifest_path   = DATA_DIR / "embeddings_manifest.json"

if not sections_path.exists():
    print("Note: sections.parquet not found (optional for this demo).")

assert chunks_path.exists(), f"Missing: {chunks_path}"
assert embeddings_path.exists(), f"Missing: {embeddings_path}"
assert manifest_path.exists(), f"Missing: {manifest_path}"

with open(manifest_path, "r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print(json.dumps(manifest, indent=2, sort_keys=True))

## Inspect Chunking

We load the chunked sections dataset and summarize chunk counts. The example chunk text is truncated for display only.

In [ ]:
chunks_table = pq.read_table(chunks_path)
chunks_df = chunks_table.to_pandas()

total_chunks = len(chunks_df)
doc_type_counts = chunks_df.groupby("doc_type").size().reset_index(name="count")

print(f"Total chunks: {total_chunks}")
display(doc_type_counts)

example_text = chunks_df.iloc[0]["text"] if total_chunks else ""
print(example_text[:500] + ("..." if len(example_text) > 500 else ""))

## Inspect Embeddings

We verify that embeddings align with chunks and check embedding dimensionality.

In [ ]:
embeddings_table = pq.read_table(embeddings_path)
embeddings_df = embeddings_table.to_pandas()

print(f"Embeddings: {len(embeddings_df)}")
print(f"Chunks: {len(chunks_df)}")
assert len(embeddings_df) == len(chunks_df), "Embeddings count does not match chunks."

embedding_dim = embeddings_df.iloc[0]["embedding_dim"] if len(embeddings_df) else None
sample_vector = embeddings_df.iloc[0]["embedding"] if len(embeddings_df) else []
print(f"Embedding dim: {embedding_dim}")
print(f"Sample vector length: {len(sample_vector)}")
assert embedding_dim == len(sample_vector), "Embedding dimension mismatch in sample."

## Retrieval Demo

We run deterministic retrieval over the existing embeddings. Results are displayed with metadata for traceability.

In [ ]:
from genai_filings.retrieval import retrieve

query = "How is net interest margin evolving?"
results = retrieve(query=query, issuer="NU", period="2025Q2", k=5)

results_df = pd.DataFrame(results)[
    ["rank", "score", "doc_type", "source_file", "chunk_id", "text_preview"]
]
display(results_df)

## Answer Synthesis Demo

We synthesize a cited answer using only the retrieved chunks, preserving citations and provenance.

In [ ]:
from genai_filings.answering import synthesize_answer
from IPython.display import Markdown, display

# Provider-aware synthesis. The generator is switchable
# (openai | anthropic); retrieval always embeds the query with
# OpenAI, matching the stored vectors.
answer = synthesize_answer(
    query=query,
    issuer="NU",
    period="2025Q2",
    k=5,
    provider="openai",
    model="gpt-5.5",
    temperature=0.0,
    max_tokens=500,
)

print(
    f"provider={answer['provider']} | model={answer['model']} | "
    f"chunks={answer['used_chunks']} | "
    f"valid_cites={answer['citations_valid']} | "
    f"invalid_cites={answer['citations_invalid']}"
)
display(Markdown(answer["answer_markdown"]))

## Provider Comparison: OpenAI vs Anthropic

The same retrieved context and question are sent to two frontier models — OpenAI `gpt-5.5` and Anthropic `claude-opus-4-8` — through the identical `synthesize_answer` interface. Retrieval is unchanged between them (query embeddings always use the OpenAI model recorded in the manifest), so any difference comes purely from generation.

Each result reports its **valid** and **invalid** citations. An invalid citation (a source number the model used that was not provided) would signal a grounding failure; we expect that list to be empty — which is the traceability guarantee made concrete.

In [ ]:
import os
from IPython.display import Markdown, display
from genai_filings.answering import synthesize_answer

PROVIDERS = [
    ("openai", "gpt-5.5"),
    ("anthropic", "claude-opus-4-8"),
]


def run_provider(provider, model):
    # Skip Anthropic gracefully if its (optional) key is absent.
    if provider == "anthropic" and not os.getenv("ANTHROPIC_API_KEY"):
        return {
            "provider": provider, "model": model, "used_chunks": 0,
            "citations_valid": [], "citations_invalid": [],
            "error": "ANTHROPIC_API_KEY not set — skipping.",
            "answer_markdown": "_Set `ANTHROPIC_API_KEY` to enable this "
            "side of the comparison._",
        }
    return synthesize_answer(
        query=query, issuer="NU", period="2025Q2", k=5,
        provider=provider, model=model,
        temperature=0.0, max_tokens=500,
    )


for provider, model in PROVIDERS:
    r = run_provider(provider, model)
    header = (
        f"### {r['provider']} — `{r['model']}`\n\n"
        f"**chunks:** {r['used_chunks']}  \n"
        f"**valid citations:** {r['citations_valid']}  \n"
        f"**invalid citations:** {r['citations_invalid']}  \n"
        f"**error:** {r['error']}\n\n"
    )
    display(Markdown(header + r["answer_markdown"]))
    display(Markdown("---"))

### A note on determinism

The newest reasoning models (OpenAI `gpt-5.x` and Anthropic `claude-opus-4-8`) no longer accept a `temperature` parameter, so their output is **not bit-reproducible** across runs. `synthesize_answer` handles this automatically, omitting `temperature` for those models and passing it only to models that still support it.

For an **auditable, reproducible** run, use a model that still honours `temperature=0.0`, such as OpenAI `gpt-4.1` or Anthropic `claude-sonnet-4-6`. The trade-off is newest-capability versus exact reproducibility; both paths are supported.

## Traceability

- Each answer is grounded in retrieved chunks that include issuer, period, document type, source file, and chunk ID.
- Hallucination is structurally prevented by restricting the model to the provided excerpts and requiring explicit citation by source number.
- Citations map directly to chunk identifiers, allowing audit back to the original filings PDFs.

## Conclusion

This capsule demonstrates a deterministic, read-only RAG workflow for filings-based QA: inspection, retrieval, and cited synthesis. It intentionally excludes agents, memory, evaluation harnesses, or any recomputation of embeddings or indexes.